# 09 — Profil Efisiensi Model (Edge / Green AI)

**Tujuan:** mengisi Tabel efisiensi (§3.4 naskah, Poin A revisi reviewer Q1) dengan **angka nyata**:
1. **Ukuran model biner** XGBoost Model A (9 fitur) dalam KB.
2. **Latensi inferensi per flow** (µs), diukur **single-thread** (meniru edge 1 vCPU).
3. **Throughput** (flow/detik) single-thread.

**Tanpa infra AWS.** Cukup SageMaker (komputasi lokal murni). Tidak ada EC2/VPC/deployment.

**Protokol jujur pengukuran:**
- Latih Model A dengan konfigurasi identik T3 (biner, 9 fitur, z-score per dataset).
- Inferensi diukur dengan XGBoost dipaksa **1 thread** (`nthread=1`) agar merepresentasikan skenario edge 1 vCPU.
- Latensi = rata-rata atas beberapa ulangan (buang warm-up), dilaporkan mean + std.
- Catat juga spesifikasi CPU aktual (jujur) untuk transparansi.

> Jalankan di SageMaker. Output: `model_efficiency.json`.

In [ ]:
# --- Bootstrap ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost')]:
    try: importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg}'); subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

import pickle, os, json, time, platform, gc
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'   # 175k -> train
OUT_JSON='../model_efficiency.json'
MODEL_PATH='../modelA_9feat.json'
SEED=42
print('CIC:', os.path.exists(CIC_PKL), '| UNSW:', os.path.exists(UNSW_TRAIN))

In [ ]:
# --- Mapping Model A + util (identik T3) ---
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys())

def build_matrix(df, side):
    idx=0 if side=='cic' else 1
    cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON
    out=out.replace([np.inf,-np.inf],np.nan)
    out=out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float).values

In [ ]:
# --- Latih Model A (CIC, biner, z-score) ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float)
sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0)
y=(np.asarray(d['y'])!=benign).astype(int)

Xc=build_matrix(cic_df,'cic')
Xtr,Xte,ytr,yte=train_test_split(Xc,y,test_size=0.3,random_state=SEED,stratify=y)
scaler=StandardScaler().fit(Xtr); Xtr_s=scaler.transform(Xtr); Xte_s=scaler.transform(Xte)

model=XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
    learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
    random_state=SEED,tree_method='hist',n_jobs=-1)
model.fit(Xtr_s,ytr)
print('Model A dilatih. n_fitur =', Xtr_s.shape[1])

In [ ]:
# --- (1) Ukuran model biner ---
model.save_model(MODEL_PATH)
size_bytes = os.path.getsize(MODEL_PATH)
size_kb = size_bytes/1024.0
print(f'Ukuran model: {size_bytes:,} byte = {size_kb:.1f} KB')

In [ ]:
# --- (2)&(3) Latensi & throughput SINGLE-THREAD (edge 1 vCPU) ---
# Muat ulang model dengan nthread=1 agar inferensi benar-benar single-thread.
import xgboost as xgb
booster = xgb.Booster()
booster.load_model(MODEL_PATH)
booster.set_param({'nthread': 1})

# Latensi per-FLOW (batch=1) — paling relevan utk deteksi inline per aliran
N_SINGLE = 2000                      # jumlah flow diuji satu per satu
idx = np.random.RandomState(SEED).choice(len(Xte_s), N_SINGLE, replace=False)
samples = Xte_s[idx].astype(np.float32)

# warm-up (buang efek cache/JIT)
for i in range(50):
    _ = booster.predict(xgb.DMatrix(samples[i:i+1]))

lat = []
for i in range(N_SINGLE):
    dm = xgb.DMatrix(samples[i:i+1])
    t0 = time.perf_counter()
    _ = booster.predict(dm)
    lat.append((time.perf_counter()-t0)*1e6)   # mikrodetik
lat = np.array(lat)
# buang 1% outlier atas (GC/scheduler noise)
lat_trim = lat[lat <= np.percentile(lat, 99)]

lat_mean = float(lat_trim.mean()); lat_std = float(lat_trim.std()); lat_med = float(np.median(lat_trim))
print(f'Latensi per-flow (batch=1, single-thread): mean={lat_mean:.1f} us | median={lat_med:.1f} us | std={lat_std:.1f} us')

# Throughput batch (skenario stream): prediksi seluruh test, single-thread.
# PENTING: ulangi REPEAT kali agar total waktu cukup besar (hindari dt~0 -> angka meledak/artefak).
dm_all = xgb.DMatrix(Xte_s.astype(np.float32))
for _ in range(3): booster.predict(dm_all)  # warm-up
REPEAT = 20
t0=time.perf_counter()
for _ in range(REPEAT): booster.predict(dm_all)
dt=(time.perf_counter()-t0)/REPEAT
thr_batch = len(Xte_s)/dt if dt>0 else float('nan')
print(f'Throughput batch single-thread: {thr_batch:,.0f} flow/detik ({len(Xte_s):,} flow dlm {dt*1e3:.1f} ms/iter, {REPEAT} iter)')
if dt < 1e-3:
    print('  PERINGATAN: dt sangat kecil -> throughput batch bisa tak-andal; utamakan throughput inkremental di bawah.')

# Throughput inkremental (1/lat_mean) — paling representatif utk deteksi inline per-flow
thr_incremental = 1e6/lat_mean
print(f'Throughput inkremental (1/latensi, per-flow): {thr_incremental:,.0f} flow/detik  <-- dipakai di naskah')

In [ ]:
# --- Spesifikasi lingkungan (transparansi) + simpan ---
import multiprocessing
env = dict(
    platform=platform.platform(),
    processor=platform.processor(),
    cpu_count_logical=multiprocessing.cpu_count(),
    python=platform.python_version(),
    xgboost=xgb.__version__,
    note='Latensi & throughput diukur SINGLE-THREAD (nthread=1) untuk meniru edge 1 vCPU.'
)

meta = dict(
    deskripsi='Profil efisiensi Model A (9 fitur, biner) untuk klaim Edge/Green AI. Angka nyata.',
    model='XGBoost Model A (9 fitur, binary:logistic, depth=8, n_estimators=200)',
    size_bytes=size_bytes, size_kb=round(size_kb,1),
    latency_per_flow_us=dict(mean=round(lat_mean,1), median=round(lat_med,1), std=round(lat_std,1),
                             n_samples=int(len(lat_trim)), batch=1, single_thread=True),
    throughput_flows_per_sec=dict(batch_single_thread=round(thr_batch),
                                  incremental=round(thr_incremental)),
    environment=env,
)
with open(OUT_JSON,'w') as f: json.dump(meta,f,indent=2)
print(json.dumps(meta, indent=2))
print('\nSaved:', OUT_JSON)
print('\nUntuk naskah Tabel efisiensi (§3.4):')
print(f'  Ukuran model biner (Model A, 9 fitur) : {size_kb:.1f} KB')
print(f'  Rata-rata latensi inferensi per flow  : {lat_mean:.1f} us (single-thread)')
print(f'  Throughput (flow/detik, 1 vCPU)       : {thr_batch:,.0f} (batch) / {thr_incremental:,.0f} (inkremental)')

## Simpan artefak deployment ke S3 (untuk T10 EC2)

EC2 (Analyzer) memerlukan **dua** artefak untuk pipeline real-traffic:
1. `modelA_9feat.json` — model XGBoost (sudah disimpan di atas).
2. `deploy_meta_9feat.json` — scaler z-score (mean & scale) + urutan fitur, agar EC2
   menormalisasi fitur NFStream **sama persis** dengan distribusi latih offline.

Keduanya diunggah ke `s3://ssh-detection-features-232032302717/unsw-far/models/`
(bucket sama dengan NIDS-01). EC2 tinggal `aws s3 cp ... --recursive` (lihat runbook).

In [ ]:
# --- Simpan deploy_meta (scaler 9 fitur) + upload model & meta ke S3 ---
import subprocess
S3_BUCKET = 'ssh-detection-features-232032302717'
S3_PREFIX = 'unsw-far/models'
META_PATH = '../deploy_meta_9feat.json'

# scaler di-fit di sel training (StandardScaler pada Xtr). Simpan mean & scale.
deploy_meta = dict(
    model='XGBoost Model A (9 fitur, binary:logistic)',
    features=CANON,                       # urutan fitur kanonik (WAJIB sama di EC2)
    scaler_mean=[float(v) for v in scaler.mean_],
    scaler_scale=[float(v) for v in scaler.scale_],
    label='biner: 0=normal, 1=attack',
)
with open(META_PATH, 'w') as f:
    json.dump(deploy_meta, f, indent=2)
print('Saved:', META_PATH)
print(json.dumps(deploy_meta, indent=2)[:400], '...')

# Upload model + meta ke S3 (butuh kredensial/role dgn akses s3 di SageMaker)
for src in [MODEL_PATH, META_PATH]:
    dst = f's3://{S3_BUCKET}/{S3_PREFIX}/{os.path.basename(src)}'
    r = subprocess.run(['aws','s3','cp',src,dst], capture_output=True, text=True)
    print(('OK   ' if r.returncode==0 else 'GAGAL')+f' {src} -> {dst}')
    if r.returncode != 0:
        print('  stderr:', r.stderr.strip())
        print('  (jika gagal: pastikan role SageMaker punya izin s3:PutObject ke bucket ini,')
        print('   atau upload manual: aws s3 cp', src, dst, ')')

print('\nVerifikasi:  aws s3 ls s3://%s/%s/' % (S3_BUCKET, S3_PREFIX))